In [1]:
import pandas as pd
import fastparquet

In [2]:
# Arquivos de Populacao

pop10 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a10anos-RIPSA.xlsx')
pop12 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-0a12anos-RIPSA.xlsx')
pop11a59 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-11a59-RIPSA.xlsx')
pop60 = pd.read_excel('Dados-apoio/proj-2015-2019-POP-60+RIPSA.xlsx')
pop_geral = pd.read_excel('Dados-apoio/proj-2015-2019-POP-GERAL-RIPSA.xlsx')

pops = [pop12,pop11a59,pop60,pop_geral]

In [5]:
####################################################################################################################
# Carrega os bancos Basico (com distancias e tempos), municipio por CIR e RAS e Sinan
base01 = pd.read_excel('Dados-iniciais/base01.xlsx')
muni_cir = pd.read_excel('Dados-iniciais/_Muni_por_Macro_DRS_CIR.xlsx')
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2015, 2020)))] # Com filtro interno
)

# Sinan para construir variavel MG
sinan_mg = sinan

# Padroniza codigo municipio do Sinan como numero inteiro
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

# Converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    sinan_mg[col] = pd.to_numeric(sinan_mg[col], errors="coerce").fillna(0)





####################################################################################################################
# Cria variavel Moderado/Grave (MG)


# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================


# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (sinan_mg["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (sinan_mg["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (sinan_mg["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (sinan_mg["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (sinan_mg["CON_SOROTE"] == "Sim") &
        (sinan_mg["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo

# ---------------------------------------------------------
# APLICAR FILTRO
# ---------------------------------------------------------

sinan_mg_mg = sinan_mg[filtro_mg].copy()

# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA (OPCIONAL)
# ---------------------------------------------------------

sinan_mg["TOTAL_MG"] = filtro_mg.astype(object) # aceita texto e booleanos

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(sinan_mg))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")


# Recodificando valores
sinan_mg.loc[sinan_mg['TOTAL_MG'] == False, 'TOTAL_MG'] = 'Leve'
sinan_mg.loc[sinan_mg['TOTAL_MG'] == True, 'TOTAL_MG'] = 'MG'

Total de casos: 117357
Moderados/Graves qualificados: 4475
Proporção: 3.81 %


In [6]:
# Junta todos os arquivos de populacao

pop_merge = pop10.copy() # Cria uma cópia para não mexer no original

for i in pops:
    # O merge traz as colunas novas e você salva o resultado em pop_merge
    pop_merge = pop_merge.merge(
        right=i.iloc[:, [0, 2]], 
        how='left', 
        on='IBGE'
    )

In [7]:
# Junta o banco de populacoes com a base 01

df = (
    base01
    .drop_duplicates()
    .merge(
        pop_merge.drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

In [8]:
# Junta df com banco de regioes
df = (
    df.merge(
    right=muni_cir,
    how='left',
    left_on="MUNI_NOME",
    right_on="MUNI_NOME",
    indicator='merge_flag'
).copy()
)

In [13]:
# Cria colunas de totais de casos por faixa etaria
total_casos = sinan_mg['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')

In [21]:
pd.crosstab(sinan_mg['ID_MN_RESI'], sinan_mg['TOTAL_MG']).dtypes

TOTAL_MG
Leve    int64
MG      int64
dtype: object

In [ ]:
pd.pivot_table(
    data=sinan_mg,
    index='ID_MN_RESI',
    columns= 'TOTAL_MG',
    values='ID_MN_RESI',
    aggfunc='size',
    fill_value=0
)

In [17]:
sinan_mg.columns

Index(['DT_SIN_PRI', 'SEM_PRI', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO',
       'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'ID_MN_RESI', 'ID_OCUPA_N',
       'ANT_DT_ACI', 'ANT_UF', 'ANT_MUNIC_', 'SG_UF', 'ANT_TEMPO_',
       'ANT_LOCA_1', 'MCLI_LOCAL', 'CLI_DOR', 'CLI_EDEMA', 'CLI_EQUIMO',
       'CLI_NECROS', 'CLI_LOCAL_', 'CLI_LOCA_1', 'MCLI_SIST', 'CLI_NEURO',
       'CLI_HEMORR', 'CLI_VAGAIS', 'CLI_MIOLIT', 'CLI_RENAL', 'CLI_OUTR_2',
       'CLI_OUTR_3', 'CLI_TEMPO_', 'TP_ACIDENT', 'ANI_TIPO_1', 'ANI_SERPEN',
       'ANI_ARANHA', 'ANI_LAGART', 'TRA_CLASSI', 'CON_SOROTE', 'NU_AMPOLAS',
       'NU_AMPOL_1', 'NU_AMPOL_8', 'NU_AMPOL_6', 'NU_AMPOL_4', 'NU_AMPO_7',
       'NU_AMPO_5', 'NU_AMPOL_9', 'NU_AMPOL_3', 'COM_LOC', 'COM_SECUND',
       'COM_NECROS', 'COM_COMPOR', 'COM_DEFICT', 'COM_APUTAC', 'COM_SISTEM',
       'COM_RENAL', 'COM_EDEMA', 'COM_SEPTIC', 'COM_CHOQUE', 'DOENCA_TRA',
       'EVOLUCAO', 'DT_OBITO', 'DT_ENCERRA', 'DT_DIGITA', 'IDADE_TIPO',
       'IDADE_COMPLETA', 'IDADE_ANOS',

In [23]:
# Junta total de casos ao banco base
df = df.merge(
    right=total_casos,
    right_on='ID_MN_RESI',
    left_on='IBGE',
    how='left'
    )


In [14]:
df

,ACESSO_LOCAL,MULTIPLO_PESA,REGIAO,PESA,MUNI_REFERENCIADO,OBSERVACOES,LAT_MUNI,LON_MUNI,LAT_PESA,LON_PESA,...,POP60,POP_GERAL,_merge,MACRO_CODIGO,MACRO_NOME,DRS_CODIGO,DRS_NOME,CIR_CODIGO,CIR_NOME,merge_flag
0,0,0,ARACATUBA,PENAPOLIS,ALTO ALEGRE,TODOS,-21.582059,-50.166198,-21.416404,-50.064911,...,934.8,4079.8,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
1,0,0,ARACATUBA,PENAPOLIS,AVANHANDAVA,TODOS,-21.460333,-49.946516,-21.416404,-50.064911,...,1383.0,11667.2,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
2,0,0,ARACATUBA,PENAPOLIS,BARBOSA,TODOS,-21.265661,-49.951816,-21.416404,-50.064911,...,984.0,6239.0,both,3536,RRAS19,3502,DRS-02 Aracatuba,35023,Consorcios do DRS II,both
3,0,0,ARACATUBA,VALPARAISO,BENTO DE ABREU,TODOS,-21.271572,-50.811723,-21.230655,-50.861195,...,392.6,2691.6,both,3536,RRAS19,3502,DRS-02 Aracatuba,35021,Central do DRS II,both
4,0,0,ARACATUBA,CLEMENTINA,BILAC,TODOS,-21.403962,-50.474640,-21.556698,-50.446533,...,1411.2,7349.8,both,3536,RRAS19,3502,DRS-02 Aracatuba,35021,Central do DRS II,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,IPIGUA,TODOS,NaN,NaN,NaN,NaN,...,912.0,5657.2,both,3531,RRAS12,3515,DRS-15 Sao Jose do Rio Preto,35155,Sao Jose do Rio Preto,both
641,0,0,SAO JOSE DO RIO PRETO,SAO JOSE DO RIO PRETO,ONDA VERDE,TODOS,NaN,NaN,NaN,NaN,...,594.2,4430.0,both,3531,RRAS12,3515,DRS-15 Sao Jose do Rio Preto,35155,Sao Jose do Rio Preto,both
642,1,0,PIRACICABA,PIRACICABA,PIRACICABA,TODOS,NaN,NaN,NaN,NaN,...,60091.8,409618.6,both,3529,RRAS14,3510,DRS-10 Piracicaba,35103,Piracicaba,both
643,0,0,PIRACICABA,PIRACICABA,RIO DAS PEDRAS,TODOS,NaN,NaN,NaN,NaN,...,3642.0,31718.0,both,3529,RRAS14,3510,DRS-10 Piracicaba,35103,Piracicaba,both


In [ ]:
# Junta TOTAL_MG ao banco base

df = df.merge(
    right=sinan_mg['ID_MN_RESI'],
    right_on='ID_MN_RESI',
    left_on='IBGE',
    how='left'
    )

In [28]:
df.to_excel('Dados-iniciais/base02.xlsx')

In [27]:
'''
# Codigo comparacao

comparacao = (
    base01[["MUNI_REFERENCIADO"]]
    .drop_duplicates()
    .merge(
        pop_merge[["MUNI_NOME"]].drop_duplicates(),
        left_on="MUNI_REFERENCIADO",
        right_on="MUNI_NOME",
        how="left",
        indicator=True
    )
)

nao_encontrados = comparacao[comparacao["_merge"] == "left_only"]

print(nao_encontrados)
'''

'\n# Codigo comparacao\n\ncomparacao = (\n    base01[["MUNI_REFERENCIADO"]]\n    .drop_duplicates()\n    .merge(\n        pop_merge[["MUNI_NOME"]].drop_duplicates(),\n        left_on="MUNI_REFERENCIADO",\n        right_on="MUNI_NOME",\n        how="left",\n        indicator=True\n    )\n)\n\nnao_encontrados = comparacao[comparacao["_merge"] == "left_only"]\n\nprint(nao_encontrados)\n'